# 03 — Products: near-optimality (D11) and route branches (D12) → `finish`

Re-attaches to the run (`cc.load` + `cost_distances` cache HIT + `corridor_network` from the band
store), then the two wall-to-wall products the maps need: `near_optimality.tif` (**G10**) and
`branches.*` (**G9**; `n_branches` / `route_irreplaceable` per edge). `finish` rewrites the outputs
and the runs index.

In [ ]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/wolverine_refugia_connectivity/, below the repo root where
# config.py and the corridor engine modules sit. Same bootstrap as analyses/northern_connectivity/.
import sys, pathlib, json
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import numpy as np
import pandas as pd
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
for _m in (config, cp, cg, cc):
    importlib.reload(_m)

KEY = "wolverine"
CFG = config.CORRIDORS[KEY]
HERE = ROOT / "analyses" / "wolverine_refugia_connectivity"

In [ ]:
RUN = "v2_run001"          # <- the run created by notebook 02 (see cc.runs(KEY))
A = cc.load(config.RESULTS_DIR / CFG["results_subdir"] / RUN)
cc.resistance(A)
cc.cost_distances(A)                        # cache HIT
cc.corridor_network(A, verbose=False)       # bands from the store: seconds
print(f"{A.run_id}: {len(A.edges)} edges, corridor {int(A.corridor.sum())*A.cell_km2:,.0f} km²")

## 1 · Near-optimality surface (**G10**)

In [ ]:
cc.near_optimality(A)

## 2 · Route branches (**G9**) — route-irreplaceable vs edge-irreplaceable, always reported together

In [ ]:
cc.route_branches(A)

## 3 · Protection status (W11) re-derived on the re-attached network, then finish

In [ ]:
cc.secured_status(A)
cc.priority_surface(A)
cc.finish(A)
cc.runs(KEY)

## Done — next: `04_results.ipynb` (the record) then `05_director_package.ipynb` (the curated set).